# Notebook 3: Group Relative Policy Optimization (GRPO)

## What we're doing
We use **GRPO** — the method behind DeepSeek-R1 — to train the model to reason about
CVE severity using **verifiable rewards**. No human labels needed after setup.

## Task
**CVE severity classification with reasoning** — given a CVE description, the model must:
1. Reason through the vulnerability (Chain-of-Thought)
2. Output the severity in `<severity>LEVEL</severity>` tags

The reward function checks *both* format compliance and accuracy — exact match.

## Dataset
- Primary: NIST NVD API (real CVEs with CVSS-assigned severities)
- Fallback: synthetic CVE descriptions with known severities

## How GRPO works
```
For each prompt x:
  1. Sample G responses: {y_1, y_2, ..., y_G} from current policy
  2. Score each: r_i = reward_function(y_i, ground_truth)
  3. Normalize scores within the group:
       A_i = (r_i - mean(r)) / std(r)  ← group-relative advantage
  4. Update policy to increase prob of high-advantage responses:
       L = -E[ A_i * log π_θ(y_i|x) ] + β * KL(π_θ ∥ π_ref)
```

**Key insight vs RLHF**: No separate reward model trained on human rankings.
The reward function IS the ground truth check. Works perfectly for tasks with
verifiable answers: math, code, structured classification.

## Reward function design
```python
def reward(response, answer):
    format_score = 1.0 if '<severity>' in response else 0.0
    accuracy_score = 2.0 if extracted_level == answer else 0.0
    return format_score + accuracy_score   # max=3.0
```

## CPU config
- G=2 samples per prompt (normally G=8, but CPU budget)
- max_seq_len=128, 1 epoch, ~100 prompts
- expect ~60–120 min on CPU

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'trl>=0.8', 'datasets', 'accelerate', 'torch'
])
print('Dependencies ready.')

In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import os, sys, re, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig

from data.loader import load_grpo_dataset, train_val_split
from shared.model_utils import (
    load_model_and_tokenizer, load_saved_model,
    generate_response, LossTracker, save_model
)

os.environ['CUDA_VISIBLE_DEVICES'] = ''
torch.manual_seed(42)
print(f'PyTorch {torch.__version__} | cpu')

In [ ]:
# ── Cell 3: Load CVE dataset ─────────────────────────────────────────────────
MAX_SAMPLES = 100   # keep small for CPU

raw_data = load_grpo_dataset(max_samples=MAX_SAMPLES)
train_data, val_data = train_val_split(raw_data, val_ratio=0.1)

# GRPO trainer expects 'prompt' column + we store 'answer' for reward function
train_hf = Dataset.from_list(train_data)
val_hf   = Dataset.from_list(val_data)

print(f'Train: {len(train_data)} | Val: {len(val_data)}')
print('\nSample:')
print('PROMPT:', train_data[0]['prompt'][:150], '...')
print('ANSWER:', train_data[0]['answer'])

# Check severity distribution
from collections import Counter
dist = Counter(d['answer'] for d in raw_data)
print('\nSeverity distribution:', dict(dist))

In [ ]:
# ── Cell 4: Define reward functions ──────────────────────────────────────────
#
# This is the HEART of GRPO — the reward function is what makes it trainable
# without human feedback. It must be:
#   1. Scalar (single number per response)
#   2. Deterministic (same input → same reward)
#   3. Aligned with the task objective
#
# We use TWO reward functions:
#   - format_reward: does the response use <severity> tags?
#   - accuracy_reward: is the severity level correct?

SEVERITY_LEVELS = {'CRITICAL', 'HIGH', 'MEDIUM', 'LOW'}

def extract_severity(text: str):
    """Extract severity from <severity>LEVEL</severity> tags."""
    match = re.search(r'<severity>(.*?)</severity>', text, re.IGNORECASE)
    if match:
        level = match.group(1).strip().upper()
        return level if level in SEVERITY_LEVELS else None
    return None


def format_reward_fn(completions, **kwargs):
    """
    Reward for correct output format.
    Returns +1.0 if <severity>...</severity> tags present, else 0.0
    """
    rewards = []
    for completion in completions:
        text = completion[0]['content'] if isinstance(completion, list) else completion
        has_tag = bool(re.search(r'<severity>.*?</severity>', text, re.IGNORECASE))
        rewards.append(1.0 if has_tag else 0.0)
    return rewards


def accuracy_reward_fn(completions, answer, **kwargs):
    """
    Reward for correct severity classification.
    Returns +2.0 if correct, 0.0 if wrong, -0.5 if format missing.
    """
    rewards = []
    for completion in completions:
        text = completion[0]['content'] if isinstance(completion, list) else completion
        predicted = extract_severity(text)
        if predicted is None:
            rewards.append(-0.5)  # penalise missing format
        elif predicted == answer:
            rewards.append(2.0)   # correct
        else:
            rewards.append(0.0)   # wrong but well-formatted
    return rewards


# Test reward functions
test_responses = [
    [{'content': 'This is a buffer overflow. <severity>CRITICAL</severity>'}],
    [{'content': 'This is HIGH severity. <severity>HIGH</severity>'}],
    [{'content': 'The severity is medium.'}],  # no tags
]

print('=== Reward function test ===')
fmt_rewards = format_reward_fn(test_responses)
acc_rewards = accuracy_reward_fn(test_responses, answer='CRITICAL')
for i, (f, a) in enumerate(zip(fmt_rewards, acc_rewards)):
    total = f + a
    print(f'  Response {i+1}: format={f:.1f}, accuracy={a:.1f}, total={total:.1f}')

In [ ]:
# ── Cell 5: Visualize GRPO advantage computation ─────────────────────────────
#
# This shows what happens inside one GRPO step.
# G=4 samples, different rewards → normalized advantages.

import numpy as np

def compute_grpo_advantages(rewards, eps=1e-8):
    """Group-relative advantage: (r_i - mean) / std"""
    rewards = np.array(rewards)
    mean_r = rewards.mean()
    std_r  = rewards.std() + eps
    return (rewards - mean_r) / std_r

# Simulate G=4 samples with different reward profiles
example_rewards = [3.0, 2.0, 0.0, -0.5]   # one correct, one formatted, two wrong
advantages = compute_grpo_advantages(example_rewards)

print('=== GRPO advantage normalization example ===')
print(f'  Raw rewards:   {example_rewards}')
print(f'  Mean reward:   {np.mean(example_rewards):.3f}')
print(f'  Std reward:    {np.std(example_rewards):.3f}')
print(f'  Advantages:    {[round(a,3) for a in advantages]}')
print()
print('  Interpretation:')
for i, (r, a) in enumerate(zip(example_rewards, advantages)):
    direction = 'increase prob' if a > 0 else 'decrease prob'
    print(f'    Sample {i+1}: reward={r:+.1f} → advantage={a:+.3f} → {direction}')

print('\n  Key insight: only RELATIVE rewards matter.')
print('  Adding +10 to all rewards → advantages unchanged → no learning signal!')

In [ ]:
# ── Cell 6: Load model ───────────────────────────────────────────────────────
sft_path = PROJECT_ROOT / 'models' / 'sft_cybersec'
if sft_path.exists():
    print('Loading SFT checkpoint for GRPO...')
    model, tokenizer = load_saved_model('sft_cybersec')
else:
    print('Loading base Qwen2.5-0.5B (run 01_sft.ipynb first for better results)...')
    model, tokenizer = load_model_and_tokenizer()

print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params')

In [ ]:
# ── Cell 7: Baseline before GRPO ─────────────────────────────────────────────
test_prompt = train_data[0]['prompt']
test_answer = train_data[0]['answer']

print('=== BEFORE GRPO ===')
print(f'Ground truth severity: {test_answer}')
resp_before = generate_response(model, tokenizer, test_prompt, max_new_tokens=80)
print(f'Response: {resp_before}')
predicted = extract_severity(resp_before)
print(f'Extracted: {predicted} | Correct: {predicted == test_answer}')

In [ ]:
# ── Cell 8: Configure GRPO training ──────────────────────────────────────────
OUTPUT_DIR = str(PROJECT_ROOT / 'models' / 'grpo_checkpoint')

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    # ── GRPO-specific ──
    num_generations=2,           # G: samples per prompt (normally 8; CPU budget=2)
    beta=0.04,                   # KL penalty (lower = more exploration)
    max_completion_length=80,
    max_prompt_length=64,
    # ── CPU training ──
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    # ── Logging ──
    logging_steps=5,
    save_steps=50,
    # ── Misc ──
    fp16=False,
    bf16=False,
    report_to='none',
    dataloader_num_workers=0,
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    reward_funcs=[format_reward_fn, accuracy_reward_fn],
    train_dataset=train_hf,
    eval_dataset=val_hf,
    processing_class=tokenizer,
)

print('Starting GRPO training...')
print(f'  Samples per prompt (G): {grpo_config.num_generations}')
print(f'  Reward functions: format + accuracy (max total reward=3.0)')
print(f'  β (KL weight): {grpo_config.beta}')
print('  Each training step samples G completions, scores them, then optimizes.\n')

train_result = grpo_trainer.train()

print('\n=== GRPO training complete ===')
print(f'  Final loss:  {train_result.training_loss:.4f}')
print(f'  Train time:  {train_result.metrics["train_runtime"]:.0f}s')

In [ ]:
# ── Cell 9: Post-training evaluation ─────────────────────────────────────────
print('=== AFTER GRPO ===' )
resp_after = generate_response(model, tokenizer, test_prompt, max_new_tokens=80)
print(f'Response: {resp_after}')
predicted = extract_severity(resp_after)
print(f'Extracted: {predicted} | Correct: {predicted == test_answer}')

# Evaluate on val set
print('\n=== Val set evaluation ===')
correct, total, formatted = 0, 0, 0
for item in val_data:
    resp = generate_response(model, tokenizer, item['prompt'], max_new_tokens=80)
    pred = extract_severity(resp)
    if pred is not None:
        formatted += 1
    if pred == item['answer']:
        correct += 1
    total += 1

print(f'  Format compliance: {formatted}/{total} = {formatted/total*100:.0f}%')
print(f'  Accuracy:          {correct}/{total} = {correct/total*100:.0f}%')

In [ ]:
# ── Cell 10: Save ─────────────────────────────────────────────────────────────
save_model(model, tokenizer, 'grpo_cybersec')

tracker = LossTracker()
for log in grpo_trainer.state.log_history:
    if 'loss' in log and 'step' in log:
        tracker.record(log['step'], log['loss'])

tracker.save(str(PROJECT_ROOT / 'models' / 'grpo_loss_history.json'))
tracker.plot_ascii('GRPO Training Loss')

print('Notebook 3 complete. Model saved to models/grpo_cybersec')

## Key takeaways

1. **Verifiable rewards are powerful**: CVE severity, code correctness, math answers — any task where you can check the output programmatically is a GRPO candidate. No human labelling budget needed after setup.

2. **Format rewards first**: training the model to emit structured output (the `<severity>` tags) is prerequisite to accuracy learning. The two reward functions work together — format compliance unlocks accuracy scoring.

3. **Group normalization is key**: GRPO only learns from *relative differences* within a group. If all G samples get reward=0, advantage=0, no update. This is why task difficulty should be calibrated so the model succeeds ~30-70% of the time.

4. **Why G matters**: larger G gives better advantage estimates (lower variance). G=2 (our CPU budget) means high variance; G=8 is the typical minimum for real training.

5. **Next**: `04_rlhf.ipynb` — RLHF with a trained reward model and PPO update loop.